# SmritiMeds Claude Vision smoke test

Upload a single medication label image, send it to Claude Vision, and inspect the raw JSON response.

In [ ]:
import base64
import json
import mimetypes
import os
from pathlib import Path

import requests

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
IMAGE_PATH = Path(os.environ.get('SMRITIMEDS_TEST_IMAGE', PROJECT_ROOT / 'sample_label.jpg'))
MODEL = os.environ.get('SMRITIMEDS_MODEL', 'claude-sonnet-4-20250514')
API_KEY = os.environ.get('ANTHROPIC_API_KEY')
assert API_KEY, 'Set ANTHROPIC_API_KEY before running this notebook.'
assert IMAGE_PATH.exists(), f'Missing test image: {IMAGE_PATH}'


In [ ]:
def encode_image(path: Path) -> tuple[str, str]:
    mime_type = mimetypes.guess_type(path.name)[0] or 'image/jpeg'
    data = base64.b64encode(path.read_bytes()).decode('utf-8')
    return mime_type, data

SYSTEM_PROMPT = '''You are SmritiMeds, a medication adherence assistant.
Return JSON only. Do not provide independent medical advice.
If the image is unclear, set needs_manual_review to true.'''

USER_PROMPT = '''Return exactly one JSON object with keys medication_name, strength, instructions_raw, times_per_day, schedule, pill_appearance, verification_summary, confidence_notes, needs_manual_review.
Use null for uncertain strings, 0 for unknown times_per_day, and [] for unknown schedule.'''

mime_type, encoded = encode_image(IMAGE_PATH)
payload = {
    'model': MODEL,
    'max_tokens': 1400,
    'temperature': 0.1,
    'system': SYSTEM_PROMPT,
    'messages': [
        {'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': mime_type, 'data': encoded}},
            {'type': 'text', 'text': USER_PROMPT},
        ]}
    ]
}
response = requests.post(
    'https://api.anthropic.com/v1/messages',
    headers={
        'x-api-key': API_KEY,
        'anthropic-version': '2023-06-01',
        'content-type': 'application/json',
    },
    json=payload,
    timeout=90,
)
response.raise_for_status()
data = response.json()
raw_text = '\n'.join(block.get('text', '') for block in data.get('content', []) if block.get('type') == 'text').strip()
print(raw_text)
